References:

[fetch all pdf files function from this notebook from a github page, which is from a youtube tutoyial](https://github.com/Jcharis/DataScienceTools/blob/master/PyPDF_CrashCourse/PyPDF2%20Crash%20Course.ipynb)

[pypdf merging pdfs](https://pypdf.readthedocs.io/en/stable/user/merging-pdfs.html)

[pypdf's guide to transforming several of copies of the same page](https://pypdf.readthedocs.io/en/stable/user/cropping-and-transforming.html#transforming-several-copies-of-the-same-page) (ended up not using it, since i moved onto pymupdf, but good to take note of this!!)

[pymupdf's guide to combining single pages](https://pymupdf.readthedocs.io/en/latest/the-basics.html#combining-single-pages)

[pymupdf's rect documentation](https://pymupdf.readthedocs.io/en/latest/rect.html) (had to understand this since I wanted to make a 2up instead of 4up as in the example above!)

Unfortunately, the functions for renaming the outputs from [bookbinder app](https://bookbinder.app/) is AI-sloppa (deep-seek). As stated in the cell that contains it, I was too tired to figure it out and none of the intricate requirements I needed were in any tutorial help I found on google/stackexchange, so I gave it a shot and it fortunately didn't have much problem figuring it out.


How to use:
make your A5 book in [bookbinder app](https://bookbinder.app/), 2 pages per side and scaled pages. Sheets per signature can technically be how many as you want, but I personally usually use 5.

python dependencies: pypdf, pymupdf. os and re (for renaming bookbinder app output) should come preinstalled.


**Beware** that some cells will give some errors as they handle the pdfs. That doesn't matter at all, and if there is a way to fix these errors, please suggest so! I am nothing but a novice trying to do something with this for my own entertainment.

In [ ]:
import pypdf as pdf
from pypdf import PdfReader, PdfWriter

In [ ]:
book_name = "your_book_name"
dir_path = "/path/to/your/file"

In [ ]:
import os
def fetch_all_pdf_files(parent_folder: str):
    target_files = []
    for path, subdirs, files in os.walk(parent_folder):
        for name in files:
            if name.endswith(".pdf"):
                target_files.append(os.path.join(path, name))
    return target_files 

In [ ]:
# sloppa code,  unfort not 5head enough (yet) and too tired to do this one. AT LEAST IT WORKED IT OUT FAST AND SUCH.
import os
import re

def natural_sort_key(filename):
    """
    Convert string into a list of string and number chunks for natural sorting.
    Handles zero-padded numbers by converting them to integers.
    e.g., 'signature00_back' -> ['signature', 0, '_back']
    """
    def convert(text):
        return int(text) if text.isdigit() else text.lower()
    
    return [convert(c) for c in re.split(r'(\d+)', filename)]

def rename_signatures(directory):
    """
    Rename signature files with zero-padded numbers in the specified directory.
    Maps signatureX_front -> odd numbers, signatureX_back -> even numbers
    """
    # Get all files in the directory
    files = [f for f in os.listdir(directory) if f.startswith('signature')]
    
    # Sort naturally (by the numeric value, not string)
    files.sort(key=natural_sort_key)
    
    # The natural sort gives us: back, front for each number
    # We want: front, back for each number
    # So let's reorder them by swapping each pair
    reordered_files = []
    for i in range(0, len(files), 2):
        if i+1 < len(files):
            # Swap each pair to put front first
            reordered_files.append(files[i+1])  # front
            reordered_files.append(files[i])    # back
        else:
            reordered_files.append(files[i])
    
    print("Original order (natural sort):")
    for i, f in enumerate(files):
        print(f"  {i+1}: {f}")
    
    print("\nReordered order (front then back):")
    for i, f in enumerate(reordered_files):
        print(f"  {i+1}: {f}")
    
    # Rename files
    print("\nRenaming:")
    for i, old_name in enumerate(reordered_files):
        # Calculate new name: start from 1
        new_number = i + 1
        # Get file extension (if any)
        name, ext = os.path.splitext(old_name)
        new_name = f"{new_number}{ext}"
        
        old_path = os.path.join(directory, old_name)
        new_path = os.path.join(directory, new_name)
        
        # Check if target already exists
        if os.path.exists(new_path):
            print(f"  Warning: {new_name} already exists, skipping {old_name}")
            continue
        
        os.rename(old_path, new_path)
        print(f"  {old_name} -> {new_name}")
    
    print(f"\nDone! Renamed {len(reordered_files)} files.")

# Recommended: explicit version that handles zero-padded numbers
def rename_signatures_explicit(directory):
    """
    More explicit version: group by signature number (handles zero-padded numbers)
    """
    # Get all files
    files = [f for f in os.listdir(directory) if f.startswith('signature')]
    
    # Group files by signature number
    signature_groups = {}
    for f in files:
        # Extract the number between 'signature' and '_'
        # This handles both 'signature0' and 'signature00' formats
        match = re.search(r'signature(\d+)_(front|back)', f)
        if match:
            # Convert to int to handle zero-padding (00 becomes 0)
            num = int(match.group(1))
            type_ = match.group(2)
            if num not in signature_groups:
                signature_groups[num] = {}
            signature_groups[num][type_] = f
    
    # Sort by signature number (numerical order)
    sorted_numbers = sorted(signature_groups.keys())
    
    # Rename in order: front first, then back
    new_number = 1
    print("Renaming:")
    for num in sorted_numbers:
        # Rename front first (odd numbers)
        if 'front' in signature_groups[num]:
            old_name = signature_groups[num]['front']
            old_path = os.path.join(directory, old_name)
            name, ext = os.path.splitext(old_name)
            new_name = f"{new_number}{ext}"
            new_path = os.path.join(directory, new_name)
            
            if os.path.exists(new_path):
                print(f"  Warning: {new_name} already exists, skipping {old_name}")
            else:
                os.rename(old_path, new_path)
                print(f"  {old_name} -> {new_name}")
            new_number += 1
        
        # Then rename back (even numbers)
        if 'back' in signature_groups[num]:
            old_name = signature_groups[num]['back']
            old_path = os.path.join(directory, old_name)
            name, ext = os.path.splitext(old_name)
            new_name = f"{new_number}{ext}"
            new_path = os.path.join(directory, new_name)
            
            if os.path.exists(new_path):
                print(f"  Warning: {new_name} already exists, skipping {old_name}")
            else:
                os.rename(old_path, new_path)
                print(f"  {old_name} -> {new_name}")
            new_number += 1
    
    print(f"\nDone! Renamed {new_number - 1} files.")

# Alternative: if you want to keep the zero-padded format in the output (though you said you want 1, 2, 3...)
def rename_signatures_keep_padding(directory):
    """
    Alternative: if you want to rename but keep zero-padded format in output
    """
    files = [f for f in os.listdir(directory) if f.startswith('signature')]
    
    # Group by signature number
    signature_groups = {}
    for f in files:
        match = re.search(r'signature(\d+)_(front|back)', f)
        if match:
            num = int(match.group(1))
            type_ = match.group(2)
            if num not in signature_groups:
                signature_groups[num] = {}
            signature_groups[num][type_] = f
    
    # Sort by signature number
    sorted_numbers = sorted(signature_groups.keys())
    
    # Rename with zero-padded numbers (e.g., 01, 02, etc.)
    new_number = 1
    # Determine padding width based on total files
    total_files = len(files)
    padding = len(str(total_files))
    
    print("Renaming:")
    for num in sorted_numbers:
        if 'front' in signature_groups[num]:
            old_name = signature_groups[num]['front']
            old_path = os.path.join(directory, old_name)
            name, ext = os.path.splitext(old_name)
            # Zero-pad the new number
            new_name = f"{str(new_number).zfill(padding)}{ext}"
            new_path = os.path.join(directory, new_name)
            
            if os.path.exists(new_path):
                print(f"  Warning: {new_name} already exists, skipping {old_name}")
            else:
                os.rename(old_path, new_path)
                print(f"  {old_name} -> {new_name}")
            new_number += 1
        
        if 'back' in signature_groups[num]:
            old_name = signature_groups[num]['back']
            old_path = os.path.join(directory, old_name)
            name, ext = os.path.splitext(old_name)
            new_name = f"{str(new_number).zfill(padding)}{ext}"
            new_path = os.path.join(directory, new_name)
            
            if os.path.exists(new_path):
                print(f"  Warning: {new_name} already exists, skipping {old_name}")
            else:
                os.rename(old_path, new_path)
                print(f"  {old_name} -> {new_name}")
            new_number += 1
    
    print(f"\nDone! Renamed {new_number - 1} files.")

# Usage
if __name__ == "__main__":
    # Replace with your actual directory path
    directory_path = dir_path
    
    # Choose which method to use:
    # rename_signatures(directory_path)           # Simple reordering method
    # rename_signatures_explicit(directory_path)    # Recommended: handles zero-padded numbers
    rename_signatures_keep_padding(directory_path)  # If you want zero-padded output (01, 02, etc.)

In [ ]:
import numpy as np
input_files = fetch_all_pdf_files(dir_path)
input_files = np.sort(input_files)
print(np.sort(input_files))

In [ ]:
# 3rd type of merge, front only, back only in separate pdfs
writer = PdfWriter()

for pdfF in input_files[0:len(input_files):2]:
    with open(pdfF,'rb') as file:
        num_pages=PdfReader(file).get_num_pages()
        writer.append(pdfF,(0,num_pages)) # front
        print('test')

writer.write("pdfF.pdf")
writer = PdfWriter()

for pdfB in input_files[1:len(input_files):2]:
    num_pages=PdfReader(pdfB).get_num_pages()
    for n in range(0,num_pages):
        writer.append(pdfB,[n]) # back

writer.write("pdfB.pdf")

In [ ]:
# final version
import pymupdf

src = pymupdf.open("pdfF.pdf") # open front only pdf
doc = pymupdf.open()

width, height = pymupdf.paper_size("a4")

r1 = pymupdf.Rect(0,0,width, height/2)  # top rect
r2 = pymupdf.Rect(0, height/2, width, height) # bottom rect
r_tab = [r1, r2]

for spage in src:
    if spage.number % 2 == 0: # create new output page
            page = doc.new_page(-1, width = width, height = height)
    page.show_pdf_page(r_tab[spage.number % 2], src, spage.number, rotate=90) # insert input page into correct rectangle rotated 90

doc.save("2upF.pdf", garbage=3, deflate=True)

src = pymupdf.open("pdfB.pdf") # open back only pdf
doc = pymupdf.open()

for spage in src:
    if spage.number % 2 == 0:
            page = doc.new_page(-1, width = width, height = height)
    page.show_pdf_page(r_tab[spage.number % 2], src, spage.number, rotate=90)

doc.save("2upB.pdf", garbage=3, deflate=True)

writer = PdfWriter()

# unlike the other time, in which i did zip for array, it only works like this now for some reason lol

x="2upF.pdf"
y="2upB.pdf"
npg=PdfReader(x).get_num_pages()
for n in range(npg):
    writer.append(x,[n])
    writer.append(y,[n])

writer.write(f"[final] {book_name}.pdf")